In [1]:
# Imports

import sys
sys.path.append("..")

import torch
from src.model import SiameseUNet
from src.loss import BCEDiceLoss, DiceLoss
from src.evaluate import MetricTracker

In [2]:
# Test loss

criterion = BCEDiceLoss(pos_weight=3.0)

logits = torch.randn(4, 1, 256, 256, requires_grad=True)
target = (torch.rand(4, 1, 256, 256) > 0.97).float()

loss = criterion(logits, target)
print(f"Combined loss: {loss.item():.4f}")

loss.backward()
print("Backward pass successful")

Combined loss: 0.8991
Backward pass successful


In [3]:
# Loss sanity check

# Perfect prediction
perfect_logits = torch.where(target == 1, torch.tensor(10.0), torch.tensor(-10.0))
print(f"Perfect prediction loss: {criterion(perfect_logits, target).item():.6f}")

# All-zeros prediction
lazy_logits = torch.full_like(target, -10.0)
print(f"All-negative loss:       {criterion(lazy_logits, target).item():.4f}")

# Inverted prediction
wrong_logits = torch.where(target == 1, torch.tensor(-10.0), torch.tensor(10.0))
print(f"Inverted prediction loss: {criterion(wrong_logits, target).item():.4f}")

Perfect prediction loss: 0.000403
All-negative loss:       0.9490
Inverted prediction loss: 5.7994


In [4]:
# Test metrics tracker

tracker = MetricTracker(threshold=0.5)

for _ in range(5):
    logits = torch.randn(4, 1, 256, 256)
    target = (torch.rand(4, 1, 256, 256) > 0.97).float()
    tracker.update(logits, target)

print(f"Random predictions: {tracker}")

Random predictions: F1=0.0569 | IoU=0.0293 | P=0.0302 | R=0.5007 | Acc=0.5007


In [5]:
# Metrics sanity check

# Perfect predictions
tracker = MetricTracker()
target = (torch.rand(4, 1, 256, 256) > 0.97).float()
perfect_logits = torch.where(target == 1, torch.tensor(10.0), torch.tensor(-10.0))
tracker.update(perfect_logits, target)
print(f"Perfect:  {tracker}")

# All-zeros prediction
tracker = MetricTracker()
tracker.update(torch.full((4, 1, 256, 256), -10.0), target)
print(f"All-neg:  {tracker}")

Perfect:  F1=1.0000 | IoU=1.0000 | P=1.0000 | R=1.0000 | Acc=1.0000
All-neg:  F1=0.0000 | IoU=0.0000 | P=0.0000 | R=0.0000 | Acc=0.9696
